# TP1 · Random Forest y validación cruzada

Este notebook evalúa Random Forest sobre las mismas siete variantes de variables de `TP1 ramiro.ipynb`. Conserva la separación inicial 80/20 y calcula validación cruzada de cinco folds sobre el 80 % de entrenamiento, con `random_state=42`.

El menor MAE medio indica mejor desempeño. El conjunto reservado del 20 % no se usa en esta comparación.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import PolynomialFeatures

pd.set_option('display.precision', 4)

In [ ]:
# Permite ejecutar desde la carpeta TP1 o desde la raíz del curso.
candidatas = [Path.cwd(), Path.cwd() / 'LdD2026 modificacion' / 'TPs' / 'TP1']
CARPETA = next((p for p in candidatas if (p / 'X_train.csv').exists()), None)
if CARPETA is None:
    raise FileNotFoundError('Ejecutá desde la carpeta TP1 o desde la raíz del curso.')

predictores = pd.read_csv(CARPETA / 'X_train.csv')
var_ind = pd.read_csv(CARPETA / 'y_train.csv')['weight']

mapas = {
    'mature': {'younger mom': 0, 'mature mom': 1},
    'sex': {'female': 0, 'male': 1},
    'habit': {'nonsmoker': 0, 'smoker': 1},
    'marital': {'not married': 0, 'married': 1},
    'whitemom': {'not white': 0, 'white': 1},
}
for columna, mapa in mapas.items():
    predictores[columna + '_bin'] = predictores[columna].map(mapa)

if predictores.filter(like='_bin').isna().any().any():
    raise ValueError('Hay categorías desconocidas en las variables categóricas.')

X_entrenamiento, X_reserva, y_entrenamiento, y_reserva = train_test_split(
    predictores, var_ind, test_size=0.20, random_state=42
)
print(f'Desarrollo: {len(X_entrenamiento)} filas | Reserva sin evaluar: {len(X_reserva)} filas')

In [ ]:
# Las funciones son las mismas que en los modelos lineales 1 a 7.
def construir_X_modelo_1(df):
    return df[['fage', 'mage', 'visits', 'gained', 'sex_bin', 'habit_bin', 'marital_bin', 'whitemom_bin', 'mature_bin']]

def construir_X_modelo_2(df):
    return df[['mage', 'visits', 'gained', 'sex_bin', 'habit_bin', 'whitemom_bin']]

def construir_X_modelo_3(df):
    return df[['mage', 'gained', 'habit_bin', 'whitemom_bin']]

def construir_X_modelo_4(df):
    return df[['mage', 'gained', 'whitemom_bin']]

def construir_X_modelo_5(df):
    visitas_seguras = df['visits'].replace(0.0, 1.0)
    return pd.DataFrame({
        'visits/mage': df['visits'] / df['mage'],
        'gained': df['gained'],
        'gained/visits': df['gained'] / visitas_seguras,
        'sex_bin*gained': df['sex_bin'] * df['gained'],
        'habit_bin*gained': df['habit_bin'] * df['gained'],
        'visits*mage': df['visits'] * df['mage'],
    }, index=df.index)

def construir_X_modelo_6(df):
    return df[['mage', 'gained', 'habit_bin', 'whitemom_bin', 'marital_bin', 'sex_bin']]

def construir_X_modelo_7(df):
    poly = PolynomialFeatures(degree=2, include_bias=False)
    return pd.DataFrame(poly.fit_transform(df[['mage', 'gained']]), index=df.index)

modelos = {
    'Modelo 1': construir_X_modelo_1,
    'Modelo 2': construir_X_modelo_2,
    'Modelo 3': construir_X_modelo_3,
    'Modelo 4': construir_X_modelo_4,
    'Modelo 5': construir_X_modelo_5,
    'Modelo 6': construir_X_modelo_6,
    'Modelo 7': construir_X_modelo_7,
}

def crear_bosque():
    return RandomForestRegressor(
        n_estimators=500, min_samples_leaf=3, random_state=42, n_jobs=-1
    )

## Validación cruzada de cinco folds

Cada fold entrena un bosque nuevo y calcula el MAE en registros no vistos. En el modelo 6 se excluyen los pesos menores de 4 libras solamente del entrenamiento, igual que en el notebook original; la validación se evalúa completa.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_validacion_cruzada = {nombre: [] for nombre in modelos}
r2_validacion_cruzada = {nombre: [] for nombre in modelos}
mae_entrenamiento = {nombre: [] for nombre in modelos}

for nombre, construir_X in modelos.items():
    for indices_ent, indices_val in kf.split(X_entrenamiento):
        X_fold_ent = X_entrenamiento.iloc[indices_ent]
        X_fold_val = X_entrenamiento.iloc[indices_val]
        y_fold_ent = y_entrenamiento.iloc[indices_ent]
        y_fold_val = y_entrenamiento.iloc[indices_val]

        if nombre == 'Modelo 6':
            mascara = y_fold_ent >= 4
            X_fold_ent = X_fold_ent.loc[mascara]
            y_fold_ent = y_fold_ent.loc[mascara]

        bosque = crear_bosque()
        X_ent = construir_X(X_fold_ent)
        X_val = construir_X(X_fold_val)
        bosque.fit(X_ent, y_fold_ent)

        pred_ent = bosque.predict(X_ent)
        pred_val = bosque.predict(X_val)
        mae_entrenamiento[nombre].append(mean_absolute_error(y_fold_ent, pred_ent))
        mae_validacion_cruzada[nombre].append(mean_absolute_error(y_fold_val, pred_val))
        r2_validacion_cruzada[nombre].append(r2_score(y_fold_val, pred_val))

tabla_folds = pd.DataFrame(mae_validacion_cruzada, index=[f'Fold {i}' for i in range(1, 6)])
print('MAE por fold:')
display(tabla_folds.round(4))

resumen_cv = pd.DataFrame({
    'MAE CV medio': tabla_folds.mean(),
    'Desvío estándar': tabla_folds.std(),
    'MAE mínimo': tabla_folds.min(),
    'MAE máximo': tabla_folds.max(),
    'R² CV medio': pd.Series(r2_validacion_cruzada).apply(np.mean),
    'MAE entrenamiento medio': pd.Series(mae_entrenamiento).apply(np.mean),
}).sort_values('MAE CV medio')

print('Resumen de Random Forest:')
display(resumen_cv.round(4))

In [ ]:
plt.figure(figsize=(11, 6))
nombres = list(modelos.keys())
datos_mae = [mae_validacion_cruzada[nombre] for nombre in nombres]
plt.boxplot(datos_mae, tick_labels=nombres, showmeans=True,
            meanprops=dict(marker='D', markerfacecolor='crimson', markeredgecolor='crimson'))
plt.title('MAE por variante de Random Forest en validación cruzada (5 folds)')
plt.xlabel('Variante de variables')
plt.ylabel('MAE (libras)')
plt.tight_layout()
plt.show()

mejor_modelo = resumen_cv.index[0]
print(f'Mejor variante según MAE CV medio: {mejor_modelo}')